# Analisis de Sentimientos en reseñas de canciones digitales



## Librerias

In [41]:
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings

warnings.filterwarnings("ignore")

nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/jhonattan.reales/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## Datos

In [42]:
# Descargamos el conjunto de reseñas desde nuestro repositorio
!wget -O Digital_Music_5.json https://raw.githubusercontent.com/Can0land/NLP-Laboratory/refs/heads/main/Digital_Music_5.json

--2026-02-14 23:57:32--  https://raw.githubusercontent.com/Can0land/NLP-Laboratory/refs/heads/main/Digital_Music_5.json
Resolviendo raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Conectando con raw.githubusercontent.com (raw.githubusercontent.com)[185.199.108.133]:443... conectado.
Petición HTTP enviada, esperando respuesta... 200 OK
Longitud: 74747848 (71M) [text/plain]
Grabando a: «Digital_Music_5.json»

Digital_Music_5.jso 100%[===================>]  71.28M  5.87MB/s    en 13s     

2026-02-14 23:57:45 (5.39 MB/s) - «Digital_Music_5.json» guardado [74747848/74747848]



Cargamos el dataset de reseñas, teniendo en cuenta que es un archivo json

In [43]:
df = pd.read_json("./Digital_music_5.json", lines=True)
df.head(3)

,overall,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,image
0,5,3.0,True,"06 3, 2013",A2TYZ821XXK2YZ,3426958910,{'Format:': ' Audio CD'},Garrett,"This is awesome to listen to, A must-have for ...",Slayer Rules!,1370217600,NaN
1,5,NaN,True,"10 11, 2014",A3OFSREZADFUDY,3426958910,{'Format:': ' Audio CD'},Ad,bien,Five Stars,1412985600,NaN
2,5,NaN,True,"02 11, 2014",A2VAMODP8M77NG,3426958910,{'Format:': ' Audio CD'},JTGabq,It was great to hear the old stuff again and I...,SLAYER!!!!!!!!!!!!!!!!!!!!!,1392076800,NaN


Eliminamos columnas que no son necesarias para nuestro analisis de sentimientos

In [44]:
df_music = df.drop(
    [
        "reviewTime",
        "unixReviewTime",
        "reviewerID",
        "image",
        "asin",
        "reviewerName",
        "verified",
    ],
    axis=1,
)
df_music.sample(10)

,overall,vote,style,reviewText,summary
169591,5,NaN,{'Format:': ' MP3 Music'},Good song!,Five Stars
22806,5,NaN,{'Format:': ' MP3 Music'},Always loved this song,Five Stars
159696,5,NaN,{'Format:': ' MP3 Music'},Love music!,Five Stars
5088,4,NaN,{'Format:': ' MP3 Music'},It's music I needed,It's music I needed
47732,4,NaN,{'Format:': ' MP3 Music'},Great Olde.,Four Stars
139198,5,NaN,{'Format:': ' MP3 Music'},I loved this song from the first time I heard ...,Clint Black is terrific!
122181,5,NaN,{'Format:': ' MP3 Music'},A MUST HAVE for Monty Python fans anywhere in ...,MUST HAVE!
120772,4,NaN,{'Format:': ' MP3 Music'},liked,Four Stars
44778,5,NaN,{'Format:': ' MP3 Music'},Wonderful song. Purchased it to play in my dau...,Great Song
49776,5,NaN,NaN,"I'm not the biggest hip hop fan, but I did lik...",cool


Nuestra tabla de reseñas ahora cuenta con las siguientes 5 columnas:

- overall: calificación general del producto (1-5)
- vote: calificación del comentario por parte de otros usuarios
- style: estilo del producto
- reviewText: texto de la reseña
- summary: resumen de la reseña

### Limpieza de datos
Verificamos la calidad de las columnas, y limpiamos los datos

In [45]:
df_music.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 169781 entries, 0 to 169780
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   overall     169781 non-null  int64  
 1   vote        7611 non-null    float64
 2   style       157989 non-null  object 
 3   reviewText  169623 non-null  object 
 4   summary     169745 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 6.5+ MB


A continuación, aplicaremos algunos pasos de limpieza

In [46]:
df_music["style"]

0          {'Format:': ' Audio CD'}
1          {'Format:': ' Audio CD'}
2          {'Format:': ' Audio CD'}
3          {'Format:': ' Audio CD'}
4          {'Format:': ' Audio CD'}
                    ...            
169776    {'Format:': ' MP3 Music'}
169777    {'Format:': ' MP3 Music'}
169778    {'Format:': ' MP3 Music'}
169779    {'Format:': ' MP3 Music'}
169780    {'Format:': ' MP3 Music'}
Name: style, Length: 169781, dtype: object

In [ ]:
# Eliminamos todas las filas con valores nulos en las columnas style y revieText
df_music = df_music.dropna(subset=["style", "reviewText"])

# Modificamos la columna style para que solo tengas valores tipo texto, sin diccionario
df_music["style"] = df_music["style"].apply(lambda x: x.get("Format"))

# A la columna vote, los valores nulos los cambiamos por 0
df_music["vote"] = df_music["vote"].fillna(0)

# las columnas tipo texto, le cambiamos las mayusculas por minusculas y le quitamos espamos en blanco al princpio o al final
df_music["reviewText"] = df_music["reviewText"].str.lower().str.strip()
df_music["summary"] = df_music["summary"].str.lower().str.strip()

In [ ]:
df_music.sample(2)

,overall,vote,style,reviewText,summary
33238,5,NaN,None,"As with all his songs, this guy tells the full...",life set to music


## EDA
Realizar un EDA para entender el comportamiento y posibles valores de cada variable.


### Variables no tipo texto